In [2]:
import polars as pl
path = "/home/jovyan/work/data/"

In [2]:
!ls $path

bronze	gold  raw  silver


In [2]:
fraud_data_df = (
    pl.scan_parquet(f"{path}bronze/Polars/complete_fraud_data.parquet")
    .select(
        pl.all().exclude([
            "mcc", "per_capita_income", "current_age",
            "card_number", "cvv", "year_pin_last_changed","card_on_dark_web",
            "address", "merchant_id", "merchant_state", "merchant_city", "zip",
            "client_latitude", "client_longitude",
        ])
    )
    .sort(["client_id", "card_id", "date", "id"])
    .with_columns(
        row_id = pl.int_range(0, pl.len())
    )
    .with_columns(
        date = pl.col("date") + pl.duration(microseconds=pl.col("row_id"))
    )  
)

fraud_data_df.show(5)

id,date,client_id,card_id,amount,use_chip,errors,merchant_latitude,merchant_longitude,mcc_description,retirement_age,birth_year,birth_month,gender,yearly_income,total_debt,credit_score,num_credit_cards,card_brand,card_type,expires,has_chip,num_cards_issued,credit_limit,acct_open_date,target,row_id
str,datetime[μs],str,str,f64,cat,cat,f64,f64,str,i64,i64,i64,cat,f64,f64,i64,i64,cat,cat,date,bool,i64,f64,date,bool,i64
"""9097921""",2011-02-01 07:12:00,"""0""","""1271""",81.1,"""Swipe Transaction""",null,43.7719,-70.6396,"""Grocery Stores, Supermarkets""",69,1986,3,"""Male""",59613.0,36199.0,763,4,"""Mastercard""","""Debit""",2021-04-01,true,2,31490.0,2011-02-01,false,0
"""9099562""",2011-02-01 12:30:00.000001,"""0""","""1271""",55.0,"""Swipe Transaction""",null,43.5835,-70.3457,"""Miscellaneous Food Stores""",69,1986,3,"""Male""",59613.0,36199.0,763,4,"""Mastercard""","""Debit""",2021-04-01,true,2,31490.0,2011-02-01,null,1
"""9099585""",2011-02-01 12:34:00.000002,"""0""","""1271""",26.33,"""Swipe Transaction""",null,43.5835,-70.3457,"""Miscellaneous Food Stores""",69,1986,3,"""Male""",59613.0,36199.0,763,4,"""Mastercard""","""Debit""",2021-04-01,true,2,31490.0,2011-02-01,null,2
"""9099695""",2011-02-01 12:53:00.000003,"""0""","""1271""",-55.0,"""Swipe Transaction""",null,43.5835,-70.3457,"""Miscellaneous Food Stores""",69,1986,3,"""Male""",59613.0,36199.0,763,4,"""Mastercard""","""Debit""",2021-04-01,true,2,31490.0,2011-02-01,false,3
"""9099949""",2011-02-01 13:46:00.000004,"""0""","""1271""",3.2,"""Swipe Transaction""",null,43.5835,-70.3457,"""Grocery Stores, Supermarkets""",69,1986,3,"""Male""",59613.0,36199.0,763,4,"""Mastercard""","""Debit""",2021-04-01,true,2,31490.0,2011-02-01,false,4


In [3]:
(
    fraud_data_df  
    .sort(["client_id", "card_id", "date", "id"]) 
    .sink_parquet(f"{path}silver/Polars/fraud_data.parquet")
)

Hacemos la secuencia temporal creando funciones ventana e intervalos de confianza siguiendo la distribución de Von Mises
http://dx.doi.org/10.1016/j.eswa.2015.12.030

In [16]:
import math
from scipy import stats
from datetime import datetime
## En el artículo referenciado se utiliza la arcotangente, por tanto nosotros debemos trabajar con ángulos 
## que oscilen entre - pi y pi. Definimos una función para la normalización
PI = math.pi
def normalize_angle(angle):
    return ((angle + PI) % (2 * PI)) - PI

## Definimos la función de Haversine con arctan2 para evitar problemas derivados de arcsin en distancias cortas
R_EARTH = 6371.0088
def calculate_haversine(lat1, lon1, lat2, lon2):
    dlat = pl.col(lat2) - pl.col(lat1)
    dlon = pl.col(lon2) - pl.col(lon1)
    a = (dlat / 2).sin().pow(2) + pl.col(lat1).cos() * pl.col(lat2).cos() * (dlon / 2).sin().pow(2)
    return 2 * R_EARTH * pl.arctan2(a.sqrt(), (1 - a).sqrt())

# Calculamos los valores críticos utilizando una distribución normal si la muestra, N_sample, es mayor a 30, 
# en caso contrario utilizaremos una distribucion normal por la teoria del limite central
T_DISTRIBUTIONS = {
    90: {n: stats.t.ppf(0.95, df=n-1) for n in range(2, 31)},
    95: {n: stats.t.ppf(0.975, df=n-1) for n in range(2, 31)},
    99: {n: stats.t.ppf(0.995, df=n-1) for n in range(2, 31)}
}
Z_VALUES = {90: 1.645, 95: 1.960, 99: 2.576}
def calculate_Zt_value(N_sample: str, confidence_level: int) -> pl.Expr:
    if confidence_level not in T_DISTRIBUTIONS:
        raise ValueError("confidence_level must be 90, 95 or 99")
    z_val = Z_VALUES[confidence_level]
    t_dist = T_DISTRIBUTIONS[confidence_level]
    expression = pl.when(pl.col(N_sample) >= 31).then(z_val)    
    for n_val, t_val in t_dist.items():
        expression = expression.when(pl.col(N_sample) == n_val).then(t_val)
    return expression.otherwise(0.0)

unbiased_filter_1 = (
    pl.col("date").is_between(datetime(2010, 1, 1), datetime(2011, 1, 8)) |
    pl.col("date").is_between(datetime(2013, 7, 12), datetime(2014, 4, 8)) |
    pl.col("date").is_between(datetime(2015, 3, 23), datetime(2016, 12, 22))   
)

unbiased_filter_2 = (
    (pl.col("target").is_not_null()) & 
    (pl.col("date").is_between(datetime(2010, 1, 1), datetime(2011, 1, 8)) |
     pl.col("date").is_between(datetime(2013, 7, 19), datetime(2014, 4, 8)) |
     pl.col("date").is_between(datetime(2015, 3, 30), datetime(2016, 12, 22))
    )
)

In [16]:
temporal_features_fraud_data_df = (
    pl.scan_parquet(f"{path}silver/Polars/fraud_data.parquet")
    .select(["id", "client_id", "card_id", "date", "target"])
    .filter(unbiased_filter_1)
    .with_columns(
        hour = pl.col("date").dt.hour(),
        minute = pl.col("date").dt.minute(),
    )
    .with_columns(
        hour_of_day = (pl.col("hour") + (pl.col("minute") / 60.0)),
        theta = normalize_angle((pl.col("hour") + (pl.col("minute") / 60.0)) * (2 * PI / 24.0))
    )
    .with_columns(
        sin_theta = pl.col("theta").sin(),
        cos_theta = pl.col("theta").cos(),
    )
    # Ordenamos los datos para hacer el rolling
    .sort(["client_id", "card_id", "date"])
    # Seleccionamos una ventana de 7 días, como se recomienda en el artículo
    .rolling(index_column="date", period="7d", group_by=["client_id", "card_id"])
    .agg([
        pl.all().last(),
        (pl.len() -1).clip(lower_bound=1).alias("N_7d"),
        pl.col("sin_theta").shift(1).sum().alias("sum_sin_7d"),
        pl.col("cos_theta").shift(1).sum().alias("sum_cos_7d"),
    ])
    .filter(unbiased_filter_2)
    .with_columns(
        day_of_week = pl.col("date").dt.weekday(),
        # El weekend lo definiremos incluyendo los dias Viernes, Sábado y Domingo
        weekend = (pl.col("date").dt.weekday() >= 5),
        # La fórmula utilizada es la que se indica en el artículo
        mu_vM_7d = pl.arctan2("sum_sin_7d", "sum_cos_7d"),
        R_sq_7d = (pl.col("sum_sin_7d") / pl.col("N_7d")).pow(2) + (pl.col("sum_cos_7d") / pl.col("N_7d")).pow(2)
    )
    .with_columns(
        sigma_vM_7d = (1.0 / pl.col("R_sq_7d").clip(lower_bound=1e-6)).clip(lower_bound=1.0).log().sqrt()
    )
    .with_columns(
        SE_7D = pl.col("sigma_vM_7d") / pl.col("N_7d").sqrt()
    ) 
    .with_columns(
        me_90 = calculate_Zt_value("N_7d", 90) * pl.col("SE_7D"),
        me_95 = calculate_Zt_value("N_7d", 95) * pl.col("SE_7D"),
        me_99 = calculate_Zt_value("N_7d", 99) * pl.col("SE_7D"),
    )
    # Creamos los Intervalos de Confianza (90%, 95%, 99%) como se recomiendo en el articulo
    .with_columns(
        ic_lower_vm_90_7d = normalize_angle(pl.col("mu_vM_7d") - pl.col("me_90")),
        ic_upper_vm_90_7d = normalize_angle(pl.col("mu_vM_7d") + pl.col("me_90")),
        ic_lower_vm_95_7d = normalize_angle(pl.col("mu_vM_7d") - pl.col("me_95")),
        ic_upper_vm_95_7d = normalize_angle(pl.col("mu_vM_7d") + pl.col("me_95")),
        ic_lower_vm_99_7d = normalize_angle(pl.col("mu_vM_7d") - pl.col("me_99")),
        ic_upper_vm_99_7d = normalize_angle(pl.col("mu_vM_7d") + pl.col("me_99")),
    )
    # Creamos una etiqueta para indicar si la transacción se encuentra dentro los IC definidos antes
    .with_columns(
        is_time_in_IC_90 = 
            pl.when(pl.col("N_7d") == 1).then(True).otherwise(
                pl.when(pl.col("ic_lower_vm_90_7d") <= pl.col("ic_upper_vm_90_7d"))
                  .then(pl.col("theta").is_between(pl.col("ic_lower_vm_90_7d"), pl.col("ic_upper_vm_90_7d")))
                  .otherwise((pl.col("theta") >= pl.col("ic_lower_vm_90_7d")) | 
                             (pl.col("theta") <= pl.col("ic_upper_vm_90_7d"))
                            )
        ),
        is_time_in_IC_95 = 
            pl.when(pl.col("N_7d") == 1).then(True).otherwise(
                pl.when(pl.col("ic_lower_vm_95_7d") <= pl.col("ic_upper_vm_95_7d"))
                  .then(pl.col("theta").is_between(pl.col("ic_lower_vm_95_7d"), pl.col("ic_upper_vm_95_7d")))
                  .otherwise((pl.col("theta") >= pl.col("ic_lower_vm_95_7d")) | 
                             (pl.col("theta") <= pl.col("ic_upper_vm_95_7d"))
                            )
        ),
        is_time_in_IC_99 = 
            pl.when(pl.col("N_7d") == 1).then(True).otherwise(
                pl.when(pl.col("ic_lower_vm_99_7d") <= pl.col("ic_upper_vm_99_7d"))
                  .then(pl.col("theta").is_between(pl.col("ic_lower_vm_99_7d"), pl.col("ic_upper_vm_99_7d")))
                  .otherwise((pl.col("theta") >= pl.col("ic_lower_vm_99_7d")) | 
                             (pl.col("theta") <= pl.col("ic_upper_vm_99_7d"))
                            )
        ),
    )
    .drop(["target", "hour", "minute", "sum_sin_7d", "sum_cos_7d", "R_sq_7d", "SE_7D", "me_90", "me_95", "me_99"])
)

temporal_features_fraud_data_df.show(10)

client_id,card_id,date,id,hour_of_day,theta,sin_theta,cos_theta,N_7d,day_of_week,weekend,mu_vM_7d,sigma_vM_7d,ic_lower_vm_90_7d,ic_upper_vm_90_7d,ic_lower_vm_95_7d,ic_upper_vm_95_7d,ic_lower_vm_99_7d,ic_upper_vm_99_7d,is_time_in_IC_90,is_time_in_IC_95,is_time_in_IC_99
str,str,datetime[μs],str,f64,f64,f64,f64,u32,i8,bool,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool
"""0""","""1271""",2013-07-19 13:23:00.001906,"""13047138""",13.383333,-2.779437,-0.354291,-0.935135,14,5,true,-1.663538,1.198727,-2.230898,-1.096179,-2.355663,-0.971414,-2.628591,-0.698486,false,false,false
"""0""","""1271""",2013-07-19 20:35:00.001908,"""13048785""",20.583333,-0.894481,-0.779884,0.625923,15,5,true,-1.777568,1.205201,-2.325655,-1.22948,-2.444986,-1.110149,-2.703907,-0.851229,false,false,true
"""0""","""1271""",2013-07-20 13:18:00.001909,"""13051758""",13.3,-2.801253,-0.333807,-0.942641,14,6,true,-1.651931,1.195052,-2.217551,-1.08631,-2.341933,-0.961928,-2.614025,-0.689837,false,false,false
"""0""","""1271""",2013-07-23 06:42:00.001911,"""13063377""",6.7,1.754056,0.983255,-0.182236,11,2,false,-2.445773,1.100984,-3.047436,-1.84411,3.097761,-1.706122,2.785344,-1.393704,false,false,false
"""0""","""1271""",2013-07-23 14:15:00.001912,"""13065451""",14.25,-2.552544,-0.55557,-0.83147,11,2,false,-2.56345,1.320764,2.997968,-1.841682,2.832434,-1.676148,2.457651,-1.301365,true,true,true
"""0""","""1271""",2013-07-23 19:16:00.001913,"""13066587""",19.266667,-1.239184,-0.945519,0.325568,12,2,false,-2.561502,1.234871,3.081493,-1.921311,2.937084,-1.776902,2.614537,-1.454355,false,false,false
"""0""","""1271""",2013-07-23 19:58:00.001914,"""13066690""",19.966667,-1.055924,-0.870356,0.492424,13,2,false,-2.397142,1.253742,-3.016889,-1.777396,3.128414,-1.639514,2.823901,-1.335001,false,false,false
"""0""","""1271""",2013-07-24 12:48:00.001916,"""13069180""",12.8,-2.932153,-0.207912,-0.978148,13,3,false,-2.388084,1.172163,-2.967505,-1.808663,-3.096416,-1.679753,2.902071,-1.395055,true,true,true
"""0""","""1271""",2013-07-24 13:18:00.001917,"""13069336""",13.3,-2.801253,-0.333807,-0.942641,14,3,false,-2.457959,1.127566,-2.991638,-1.92428,-3.108996,-1.806922,2.917463,-1.550196,true,true,true


In [17]:
temporal_features_fraud_data_df.select(
    pl.len().alias("rows"),
    pl.col("id").n_unique().alias("unique_ids")
).collect()

rows,unique_ids
u32,u32
3121274,3121274


In [27]:
temporal_features_fraud_data_df.select(
    pl.all().null_count()
).collect()

client_id,card_id,date,id,hour_of_day,theta,sin_theta,cos_theta,N_7d,day_of_week,weekend,mu_vM_7d,sigma_vM_7d,ic_lower_vm_90_7d,ic_upper_vm_90_7d,ic_lower_vm_95_7d,ic_upper_vm_95_7d,ic_lower_vm_99_7d,ic_upper_vm_99_7d,is_time_in_IC_90,is_time_in_IC_95,is_time_in_IC_99
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [13]:
(
    temporal_features_fraud_data_df
    .sort(["client_id", "card_id", "date", "id"])
    .sink_parquet(f"{path}silver/Polars/temporal_features_fraud_data.parquet")
)

In [19]:
amount_features_fraud_data_df = (
    pl.scan_parquet(f"{path}silver/Polars/fraud_data.parquet")
    .select("client_id", "card_id", "date", "id", "target", 
            "amount", "credit_limit", "yearly_income", "total_debt",
           )
    .filter(unbiased_filter_1)
    .with_columns(
        log_amount = (pl.col("amount").abs() + 1.0).log()
    )
    # Ordenamos los datos para hacer el rolling
    .sort(["client_id", "card_id", "date"])
    # Seleccionamos una ventana de 24 horas
    .rolling(index_column="date", period="1d", group_by=["client_id", "card_id"])
    .agg([
        pl.all().last(),
        pl.col("amount").shift(1).sum().alias("sum_amount_24h")
    ])
    # Hacemos un segundo rollin seleccionando una ventana de 7 días
    .rolling(index_column="date", period="7d", group_by=["client_id", "card_id"])
    .agg([
        pl.all().last(),
        (pl.len() -1).clip(lower_bound=1).alias("N_7d"),
        pl.col("log_amount").shift(1).mean().fill_null(0.0).alias("mean_log_amount_7d"),
        pl.col("log_amount").shift(1).std().fill_null(0.0).alias("stddev_log_amount_7d"),
        pl.col("amount").shift(1).sum().alias("sum_amount_7d"),
    ])
    .filter(unbiased_filter_2)
    .with_columns(
        is_refund = (pl.col("amount") < 0.0),
        acceleration_ratio_24h_vs_7d = 
            (pl.col("sum_amount_24h") / pl.col("sum_amount_7d")).fill_null(0.0),
        ratio_24_credit_limit = 
            (pl.col("sum_amount_24h") / pl.col("credit_limit")).fill_null(0.0),
        ratio_24_yearly_income = 
            (pl.col("sum_amount_24h") / pl.col("yearly_income")).fill_null(0.0), 
        ratio_24_total_debt = 
            (pl.col("sum_amount_24h") / pl.col("total_debt")).fill_null(0.0)
    )
    .with_columns(
        SE_7D = pl.col("stddev_log_amount_7d") / pl.col("N_7d").sqrt()
    )
    .with_columns(
        me_90 = calculate_Zt_value("N_7d", 90) * pl.col("SE_7D"),
        me_95 = calculate_Zt_value("N_7d", 95) * pl.col("SE_7D"),
        me_99 = calculate_Zt_value("N_7d", 99) * pl.col("SE_7D"),
    )    
    # Creamos los Intervalos de Confianza (90%, 95%, 99%)
    .with_columns(
        ic_lower_amt_90_7d = pl.col("mean_log_amount_7d") - pl.col("me_90"),
        ic_upper_amt_90_7d = pl.col("mean_log_amount_7d") + pl.col("me_90"),
        ic_lower_amt_95_7d = pl.col("mean_log_amount_7d") - pl.col("me_95"),
        ic_upper_amt_95_7d = pl.col("mean_log_amount_7d") + pl.col("me_95"),
        ic_lower_amt_99_7d = pl.col("mean_log_amount_7d") - pl.col("me_99"),
        ic_upper_amt_99_7d = pl.col("mean_log_amount_7d") + pl.col("me_99"),
    )
    # Creamos una etiqueta para indicar si la transacción se encuentra dentro los IC definidos antes
    .with_columns(
        is_amt_in_ic_90 = pl.when(pl.col("N_7d") == 1).then(True).otherwise(
            (pl.col("log_amount") >= pl.col("ic_lower_amt_90_7d")) & 
            (pl.col("log_amount") <= pl.col("ic_upper_amt_90_7d"))
        ),
        is_amt_in_ic_95 = pl.when(pl.col("N_7d") == 1).then(True).otherwise(
            (pl.col("log_amount") >= pl.col("ic_lower_amt_95_7d")) & 
            (pl.col("log_amount") <= pl.col("ic_upper_amt_95_7d"))
        ),
        is_amt_in_ic_99 = pl.when(pl.col("N_7d") == 1).then(True).otherwise(
            (pl.col("log_amount") >= pl.col("ic_lower_amt_99_7d")) & 
            (pl.col("log_amount") <= pl.col("ic_upper_amt_99_7d"))
        )
    )
    .drop(["target", "log_amount",  
           "N_7d", "SE_7D", "me_90", "me_95", "me_99"])
)

amount_features_fraud_data_df.show(10)

client_id,card_id,date,id,amount,credit_limit,yearly_income,total_debt,sum_amount_24h,mean_log_amount_7d,stddev_log_amount_7d,sum_amount_7d,is_refund,acceleration_ratio_24h_vs_7d,ratio_24_credit_limit,ratio_24_yearly_income,ratio_24_total_debt,ic_lower_amt_90_7d,ic_upper_amt_90_7d,ic_lower_amt_95_7d,ic_upper_amt_95_7d,ic_lower_amt_99_7d,ic_upper_amt_99_7d,is_amt_in_ic_90,is_amt_in_ic_95,is_amt_in_ic_99
str,str,datetime[μs],str,f64,f64,f64,f64,f64,f64,f64,f64,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool
"""0""","""1271""",2013-07-19 13:23:00.001906,"""13047138""",42.52,31490.0,59613.0,36199.0,52.38,3.660233,0.861347,562.84,false,0.093064,0.001663,0.000879,0.001447,3.252556,4.06791,3.162906,4.15756,2.966793,4.353673,true,true,true
"""0""","""1271""",2013-07-19 20:35:00.001908,"""13048785""",37.51,31490.0,59613.0,36199.0,52.79,3.667118,0.831563,605.25,false,0.08722,0.001676,0.000886,0.001458,3.288949,4.045286,3.206614,4.127622,3.027964,4.306272,true,true,true
"""0""","""1271""",2013-07-20 13:18:00.001909,"""13051758""",44.17,31490.0,59613.0,36199.0,90.3,3.641438,0.859551,551.63,false,0.163697,0.002868,0.001515,0.002495,3.234611,4.048265,3.145148,4.137728,2.949444,4.333432,true,true,true
"""0""","""1271""",2013-07-23 06:42:00.001911,"""13063377""",14.5,31490.0,59613.0,36199.0,47.47,3.814873,0.60868,434.15,false,0.10934,0.001507,0.000796,0.001311,3.482243,4.147503,3.405956,4.22379,3.233235,4.396511,false,false,false
"""0""","""1271""",2013-07-23 14:15:00.001912,"""13065451""",16.73,31490.0,59613.0,36199.0,14.5,3.70915,0.687576,400.06,false,0.036245,0.00046,0.000243,0.000401,3.333405,4.084895,3.24723,4.17107,3.052122,4.366178,false,false,false
"""0""","""1271""",2013-07-23 19:16:00.001913,"""13066587""",8.74,31490.0,59613.0,36199.0,31.23,3.639659,0.698377,416.79,false,0.07493,0.000992,0.000524,0.000863,3.277601,4.001717,3.195932,4.083387,3.013516,4.265803,false,false,false
"""0""","""1271""",2013-07-23 19:58:00.001914,"""13066690""",9.73,31490.0,59613.0,36199.0,39.97,3.534781,0.768166,425.53,false,0.09393,0.001269,0.00067,0.001104,3.155063,3.914499,3.070583,3.998979,2.884008,4.185554,false,false,false
"""0""","""1271""",2013-07-24 12:48:00.001916,"""13069180""",53.57,31490.0,59613.0,36199.0,181.17,3.477671,0.852084,429.99,false,0.421335,0.005753,0.003039,0.005005,3.056471,3.898871,2.962762,3.99258,2.755806,4.199537,false,false,true
"""0""","""1271""",2013-07-24 13:18:00.001917,"""13069336""",31.37,31490.0,59613.0,36199.0,234.74,3.514943,0.830449,483.56,false,0.485441,0.007454,0.003938,0.006485,3.12189,3.907997,3.035456,3.994431,2.846378,4.183509,true,true,true


In [20]:
amount_features_fraud_data_df.select(
    pl.len().alias("rows"),
    pl.col("id").n_unique().alias("unique_ids")
).collect()

rows,unique_ids
u32,u32
3121274,3121274


In [25]:
amount_features_fraud_data_df.select(
    pl.all().null_count()
).collect()

client_id,card_id,date,id,amount,credit_limit,yearly_income,total_debt,sum_amount_24h,mean_log_amount_7d,stddev_log_amount_7d,sum_amount_7d,is_refund,acceleration_ratio_24h_vs_7d,ratio_24_credit_limit,ratio_24_yearly_income,ratio_24_total_debt,ic_lower_amt_90_7d,ic_upper_amt_90_7d,ic_lower_amt_95_7d,ic_upper_amt_95_7d,ic_lower_amt_99_7d,ic_upper_amt_99_7d,is_amt_in_ic_90,is_amt_in_ic_95,is_amt_in_ic_99
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [21]:
(
    amount_features_fraud_data_df
    .sort(["client_id", "card_id", "date", "id"])
    .sink_parquet(f"{path}silver/Polars/amount_features_fraud_data.parquet")
)

In [22]:
time_diff_features_fraud_data_df = (
    pl.scan_parquet(f"{path}silver/Polars/fraud_data.parquet")
    .select("client_id", "card_id", "date", "id", "target")
    .filter(unbiased_filter_1)
    # Ordenamos los datos para hacer el shift y después el rolling
    .sort(["client_id", "card_id", "date"])
    # Calculamos los minutos desde la última transacción
    .with_columns(
        time_diff_m = 
            (pl.col("date") - pl.col("date").shift(1).over(["client_id", "card_id"])).dt.total_minutes()
    )
    .with_columns(
        log_time_diff_m = (pl.col("time_diff_m").fill_null(0.0) + 1.0).log()
    )
    # Seleccionamos una ventana de 7 días
    .rolling(index_column="date", period="7d", group_by=["client_id", "card_id"])
    .agg([
        pl.all().last(),
        (pl.len() -1).clip(lower_bound=1).alias("N_7d"),
        pl.col("log_time_diff_m").shift(1).mean().fill_null(0.0).alias("mean_time_diff_m"),
        pl.col("log_time_diff_m").shift(1).std().fill_null(0.0).alias("stddev_time_diff_m")
    ])
    .filter(unbiased_filter_2)
    .with_columns(
        SE_7D = pl.col("stddev_time_diff_m") / pl.col("N_7d").sqrt()
    )
    .with_columns(
        me_90 = calculate_Zt_value("N_7d", 90) * pl.col("SE_7D"),
        me_95 = calculate_Zt_value("N_7d", 95) * pl.col("SE_7D"),
        me_99 = calculate_Zt_value("N_7d", 99) * pl.col("SE_7D"),
    )
    # Creamos los Intervalos de Confianza (90%, 95%, 99%)
    .with_columns(
        ic_lower_tdiffm_90 = pl.col("mean_time_diff_m") - pl.col("me_90"),
        ic_upper_tdiffm_90 = pl.col("mean_time_diff_m") + pl.col("me_90"),
        ic_lower_tdiffm_95 = pl.col("mean_time_diff_m") - pl.col("me_95"),
        ic_upper_tdiffm_95 = pl.col("mean_time_diff_m") + pl.col("me_95"),
        ic_lower_tdiffm_99 = pl.col("mean_time_diff_m") - pl.col("me_99"),
        ic_upper_tdiffm_99 = pl.col("mean_time_diff_m") + pl.col("me_99"),
    )
    # Creamos una etiqueta para indicar si la transacción se encuentra dentro los IC definidos antes
    .with_columns(
        is_tdiffm_in_IC90 = 
            pl.when(pl.col("N_7d") == 1).then(True).otherwise(
                pl.col("log_time_diff_m").is_between(
                    pl.col("ic_lower_tdiffm_90"), pl.col("ic_upper_tdiffm_90"))
        ),
        is_tdiffm_in_IC95 = 
            pl.when(pl.col("N_7d") == 1).then(True).otherwise(
                pl.col("log_time_diff_m").is_between(
                    pl.col("ic_lower_tdiffm_95"), pl.col("ic_upper_tdiffm_95"))
        ),
        is_tdiffm_in_IC99 = 
            pl.when(pl.col("N_7d") == 1).then(True).otherwise(
                pl.col("log_time_diff_m").is_between(
                    pl.col("ic_lower_tdiffm_99"), pl.col("ic_upper_tdiffm_99"))
        )
    )
    .drop(["target", "N_7d", "time_diff_m", "SE_7D", "me_90", "me_95", "me_99"])
)

time_diff_features_fraud_data_df.show(10)

client_id,card_id,date,id,log_time_diff_m,mean_time_diff_m,stddev_time_diff_m,ic_lower_tdiffm_90,ic_upper_tdiffm_90,ic_lower_tdiffm_95,ic_upper_tdiffm_95,ic_lower_tdiffm_99,ic_upper_tdiffm_99,is_tdiffm_in_IC90,is_tdiffm_in_IC95,is_tdiffm_in_IC99
str,str,datetime[μs],str,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool
"""0""","""1271""",2013-07-19 13:23:00.001906,"""13047138""",7.248504,5.526262,1.964124,4.596639,6.455886,4.392211,6.660314,3.945016,7.107509,false,false,false
"""0""","""1271""",2013-07-19 20:35:00.001908,"""13048785""",2.833213,5.643853,1.944779,4.759429,6.528277,4.566871,6.720836,4.149062,7.138644,false,false,false
"""0""","""1271""",2013-07-20 13:18:00.001909,"""13051758""",6.911747,5.48505,2.067322,4.506582,6.463517,4.291413,6.678686,3.820722,7.149377,false,false,true
"""0""","""1271""",2013-07-23 06:42:00.001911,"""13063377""",6.957497,5.535058,2.399648,4.223704,6.846412,3.922953,7.147164,3.242022,7.828094,false,true,true
"""0""","""1271""",2013-07-23 14:15:00.001912,"""13065451""",6.118097,5.548354,2.407817,4.232536,6.864172,3.93076,7.165948,3.247512,7.849196,true,true,true
"""0""","""1271""",2013-07-23 19:16:00.001913,"""13066587""",5.710427,5.595833,2.301648,4.402596,6.78907,4.133435,7.05823,3.532246,7.659419,true,true,true
"""0""","""1271""",2013-07-23 19:58:00.001914,"""13066690""",3.7612,5.604647,2.203889,4.515226,6.694069,4.272851,6.936444,3.737564,7.471731,false,false,true
"""0""","""1271""",2013-07-24 12:48:00.001916,"""13069180""",5.820083,5.428035,2.257912,4.31191,6.544161,4.063593,6.792478,3.515185,7.340886,true,true,true
"""0""","""1271""",2013-07-24 13:18:00.001917,"""13069336""",3.433987,5.456039,2.171861,4.428093,6.483984,4.202044,6.710034,3.707551,7.204527,false,false,false


In [23]:
time_diff_features_fraud_data_df.select(
    pl.len().alias("rows"),
    pl.col("id").n_unique().alias("unique_ids")
).collect()

rows,unique_ids
u32,u32
3121274,3121274


In [24]:
time_diff_features_fraud_data_df.select(
    pl.all().null_count()
).collect()

client_id,card_id,date,id,log_time_diff_m,mean_time_diff_m,stddev_time_diff_m,ic_lower_tdiffm_90,ic_upper_tdiffm_90,ic_lower_tdiffm_95,ic_upper_tdiffm_95,ic_lower_tdiffm_99,ic_upper_tdiffm_99,is_tdiffm_in_IC90,is_tdiffm_in_IC95,is_tdiffm_in_IC99
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [26]:
(
    time_diff_features_fraud_data_df
    .sort(["client_id", "card_id", "date", "id"])
    .sink_parquet(f"{path}silver/Polars/time_diff_features_fraud_data.parquet")
)

In [28]:
spatial_features_fraud_data_df = (
    pl.scan_parquet(f"{path}silver/Polars/fraud_data.parquet")
    .select("client_id", "card_id", "date", "id", "target", "merchant_latitude", "merchant_longitude")
    .filter(unbiased_filter_1)
    .with_columns(
        merchant_latitude_rad = pl.col("merchant_latitude").radians(), 
        merchant_longitude_rad = pl.col("merchant_longitude").radians()
    )
    .sort(["client_id", "card_id", "date"])
    #Las transacciones online no tienen coordenadas, para evitar errores las rellenamos con la anterior no nula
    .with_columns(
        pl.col(["merchant_latitude_rad", "merchant_longitude_rad"])
            .forward_fill().backward_fill().over(["client_id", "card_id"])
    )
    # Obtenemos las coordenadas de las transacciones previas
    .with_columns(
        prev_merchant_latitude_rad = pl.col("merchant_latitude_rad").shift(1).over(["client_id", "card_id"]),
        prev_merchant_longitude_rad = pl.col("merchant_longitude_rad").shift(1).over(["client_id", "card_id"]),
        time_diff_s = (pl.col("date") - pl.col("date").shift(1).over(["client_id", "card_id"])).dt.total_seconds()
    )
    .with_columns(
        distance_km = calculate_haversine("merchant_latitude_rad", "merchant_longitude_rad", 
                                       "prev_merchant_latitude_rad", "prev_merchant_longitude_rad").fill_null(0.0),
    )
    .with_columns(
        speed_km_s = (pl.col("distance_km") / pl.col("time_diff_s").clip(lower_bound=0.1)).fill_null(0.0)
    )
    .rolling(index_column="date", period="2h", group_by=["client_id", "card_id"])
    .agg([
        pl.all().last(),
        (pl.len() -1).clip(lower_bound=1).alias("N_2h"),
        pl.col("speed_km_s").shift(1).mean().fill_null(0.0).alias("mean_speed_km_s"),
        pl.col("speed_km_s").shift(1).std().fill_null(0.0).alias("stddev_speed_km_s")
    ])
    .filter(unbiased_filter_2)
    .with_columns(
        SE_2h = pl.col("stddev_speed_km_s") / pl.col("N_2h").sqrt()
    )
    .with_columns(
        me_90 = calculate_Zt_value("N_2h", 90) * pl.col("SE_2h"),
        me_95 = calculate_Zt_value("N_2h", 95) * pl.col("SE_2h"),
        me_99 = calculate_Zt_value("N_2h", 99) * pl.col("SE_2h"),
    )
    # Creamos los Intervalos de Confianza (90%, 95%, 99%)
    .with_columns(
        ic_lower_speed_90 = pl.col("mean_speed_km_s") - pl.col("me_90"),
        ic_upper_speed_90 = pl.col("mean_speed_km_s") + pl.col("me_90"),
        ic_lower_speed_95 = pl.col("mean_speed_km_s") - pl.col("me_95"),
        ic_upper_speed_95 = pl.col("mean_speed_km_s") + pl.col("me_95"),
        ic_lower_speed_99 = pl.col("mean_speed_km_s") - pl.col("me_99"),
        ic_upper_speed_99 = pl.col("mean_speed_km_s") + pl.col("me_99"),
    )
    # Creamos una etiqueta para indicar si la transacción se encuentra dentro los IC definidos antes
    .with_columns(
        is_tdiffm_in_IC90 = 
            pl.when(pl.col("N_2h") == 1).then(True).otherwise(
                pl.col("speed_km_s").is_between(
                    pl.col("ic_lower_speed_90"), pl.col("ic_lower_speed_90"))
        ),
        is_tdiffm_in_IC95 = 
            pl.when(pl.col("N_2h") == 1).then(True).otherwise(
                pl.col("speed_km_s").is_between(
                    pl.col("ic_lower_speed_95"), pl.col("ic_lower_speed_95"))
        ),
        is_tdiffm_in_IC99 = 
            pl.when(pl.col("N_2h") == 1).then(True).otherwise(
                pl.col("speed_km_s").is_between(
                    pl.col("ic_lower_speed_99"), pl.col("ic_lower_speed_99"))
        )
    )
    .drop(["prev_merchant_latitude_rad", "prev_merchant_longitude_rad", 
           "merchant_latitude_rad", "merchant_longitude_rad", 
           "time_diff_s", "SE_2h", "me_90", "me_95", "me_99", "target",
          ])
)

spatial_features_fraud_data_df.show(20)

client_id,card_id,date,id,merchant_latitude,merchant_longitude,distance_km,speed_km_s,N_2h,mean_speed_km_s,stddev_speed_km_s,ic_lower_speed_90,ic_upper_speed_90,ic_lower_speed_95,ic_upper_speed_95,ic_lower_speed_99,ic_upper_speed_99,is_tdiffm_in_IC90,is_tdiffm_in_IC95,is_tdiffm_in_IC99
str,str,datetime[μs],str,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool
"""0""","""1271""",2013-07-19 13:23:00.001906,"""13047138""",44.6653,-70.1329,143.23463,0.001699,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,true,true,true
"""0""","""1271""",2013-07-19 20:35:00.001908,"""13048785""",43.5835,-70.3457,0.0,0.0,1,0.004867,0.0,0.004867,0.004867,0.004867,0.004867,0.004867,0.004867,true,true,true
"""0""","""1271""",2013-07-20 13:18:00.001909,"""13051758""",44.8242,-68.7918,185.398039,0.003081,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,true,true,true
"""0""","""1271""",2013-07-23 06:42:00.001911,"""13063377""",43.5835,-70.3457,11.059665,0.000176,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,true,true,true
"""0""","""1271""",2013-07-23 14:15:00.001912,"""13065451""",null,null,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,true,true,true
"""0""","""1271""",2013-07-23 19:16:00.001913,"""13066587""",43.5835,-70.3457,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,true,true,true
"""0""","""1271""",2013-07-23 19:58:00.001914,"""13066690""",43.5835,-70.3457,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,true,true,true
"""0""","""1271""",2013-07-24 12:48:00.001916,"""13069180""",43.4715,-70.8064,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,true,true,true
"""0""","""1271""",2013-07-24 13:18:00.001917,"""13069336""",43.5835,-70.3457,39.174471,0.021764,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,true,true,true


In [29]:
spatial_features_fraud_data_df.select(
    pl.len().alias("rows"),
    pl.col("id").n_unique().alias("unique_ids")
).collect()

rows,unique_ids
u32,u32
3121274,3121274


In [30]:
spatial_features_fraud_data_df.select(
    pl.all().null_count()
).collect()

client_id,card_id,date,id,merchant_latitude,merchant_longitude,distance_km,speed_km_s,N_2h,mean_speed_km_s,stddev_speed_km_s,ic_lower_speed_90,ic_upper_speed_90,ic_lower_speed_95,ic_upper_speed_95,ic_lower_speed_99,ic_upper_speed_99,is_tdiffm_in_IC90,is_tdiffm_in_IC95,is_tdiffm_in_IC99
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,372516,372516,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [31]:
(
    spatial_features_fraud_data_df
    .sort(["client_id", "card_id", "date", "id"])
    .sink_parquet(f"{path}silver/Polars/spatial_features_fraud_data.parquet")
)

In [24]:
Metrics_OHE_fraud_data_df = (
    pl.scan_parquet(f"{path}silver/Polars/fraud_data.parquet")
    .select("client_id", "card_id", "date", "id", "target",
            "birth_year", "birth_month", "retirement_age", "acct_open_date", "expires",
            "errors", "card_brand", "card_type", "use_chip", "gender",
            "mcc_description",
    )
    .filter(unbiased_filter_2)
    .with_columns(
        pl.col("errors").cast(pl.String),
        pl.col("card_brand").cast(pl.String),
        pl.col("card_type").cast(pl.String),
        pl.col("use_chip").cast(pl.String),
        pl.col("gender").cast(pl.String),
        # En esta seccion nos dedicamos a calcular las últimas métricas temporales
        agemonth_at_event = (
            (pl.col("date").dt.year() - pl.col("birth_year")) * 12 +
            (pl.col("date").dt.month() - pl.col("birth_month"))
        ),
        days_acct_open = (pl.col("date") - pl.col("acct_open_date")).dt.total_days(),
        days_expires = (pl.col("expires") - pl.col("date")).dt.total_days()
    )
    .with_columns(
        retirement_years = pl.col("retirement_age") * 12 - pl.col("agemonth_at_event")
        
    )
    # En esta seccion hacemos One Hot Encoding para el género, el tipo de transacción 
    # los errores, la marca y tipo de tarjeta
    .with_columns(
        No_Errors = pl.col("errors").is_null(),
        Bad_Card_Number = pl.col("errors").str.contains("Bad Card Number", literal=True).fill_null(False),
        Bad_CVV = pl.col("errors").str.contains("Bad CVV", literal=True).fill_null(False),
        Bad_Expiration = pl.col("errors").str.contains("Bad Expiration", literal=True).fill_null(False),
        Bad_PIN = pl.col("errors").str.contains("Bad PIN", literal=True).fill_null(False),
        Bad_Zipcode = pl.col("errors").str.contains("Bad Zipcode", literal=True).fill_null(False),
        Insufficient_Balance  = pl.col("errors").str.contains("Insufficient Balance ", literal=True).fill_null(False),
        Technical_Glitch = pl.col("errors").str.contains("Technical Glitch", literal=True).fill_null(False),

        Visa = pl.col("card_brand").str.contains("Visa", literal=True).fill_null(False),
        Discover = pl.col("card_brand").str.contains("Discover", literal=True).fill_null(False),
        Amex = pl.col("card_brand").str.contains("Amex", literal=True).fill_null(False),
        Mastercard = pl.col("card_brand").str.contains("Mastercard", literal=True).fill_null(False),

        Debit_Prepaid = pl.col("card_type").str.contains("Debit (Prepaid)", literal=True).fill_null(False),
        Credit = pl.col("card_type").str.contains("Credit", literal=True).fill_null(False),
        Debit = pl.col("card_type").str.contains("Debit", literal=True).fill_null(False),

        Online_Transaction = pl.col("use_chip").str.contains("Online Transaction", literal=True).fill_null(False),
        Chip_Transaction = pl.col("use_chip").str.contains("Chip Transaction", literal=True).fill_null(False),
        Swipe_Transaction = pl.col("use_chip").str.contains("Swipe Transaction", literal=True).fill_null(False),

        Male = pl.col("gender").str.contains("Male", literal=True),  
    )
    # En esta sección calcularemos el risk factor del tipo de comercio usando funciones ventana para evitar
    # Data Leakage, por lo que tendremos que ordenar nuestros datos por fecha
    .sort("date")
    .with_columns(
        Fraudulent = pl.when(pl.col("target")).then(1).otherwise(0),
        Legitimate = pl.when(~pl.col("target")).then(1).otherwise(0)
    )
    .with_columns(
    mcc_fraud_cum = pl.col("Fraudulent").cum_sum().over("mcc_description"),
    mcc_leg_cum = pl.col("Legitimate").cum_sum().over("mcc_description"),
    total_fraud_cum = pl.col("Fraudulent").cum_sum(),
    total_leg_cum = pl.col("Legitimate").cum_sum()
    )
    .with_columns(
        mcc_fraud_prev = pl.col("mcc_fraud_cum").shift(1).over("mcc_description").fill_null(0),
        mcc_leg_prev = pl.col("mcc_leg_cum").shift(1).over("mcc_description").fill_null(0),
        
        total_fraud_prev = pl.col("total_fraud_cum").shift(1).fill_null(0),
        total_leg_prev = pl.col("total_leg_cum").shift(1).fill_null(0)
    )    
    .with_columns(
        mcc_rate = pl.col("mcc_fraud_prev") / (pl.col("mcc_leg_prev") + 1),
        fraud_rate = pl.col("total_fraud_prev") / (pl.col("total_leg_prev") + 1)
    )
    .with_columns(
        mcc_risk_factor = pl.col("mcc_rate") / (pl.col("fraud_rate") + 0.00001)
    )
    .with_columns(
        Log2_risk_factor = (pl.col("mcc_risk_factor") + 0.00001).log(base=2).round(2)
    )
    .drop([
        "errors", "card_brand", "card_type", "use_chip", 
        "gender", "birth_year", "birth_month", "retirement_age",
        "acct_open_date", "expires", "mcc_description",
        "Fraudulent", "Legitimate", "mcc_fraud_cum", "mcc_leg_cum", 
        "total_fraud_cum", "total_leg_cum", "mcc_fraud_prev", "mcc_leg_prev", 
        "total_fraud_prev", "total_leg_prev", "mcc_rate", "fraud_rate", "mcc_risk_factor"
    ])
)

Metrics_OHE_fraud_data_df.show(5)

client_id,card_id,date,id,target,agemonth_at_event,days_acct_open,days_expires,retirement_years,No_Errors,Bad_Card_Number,Bad_CVV,Bad_Expiration,Bad_PIN,Bad_Zipcode,Insufficient_Balance,Technical_Glitch,Visa,Discover,Amex,Mastercard,Debit_Prepaid,Credit,Debit,Online_Transaction,Chip_Transaction,Swipe_Transaction,Male,Log2_risk_factor
str,str,datetime[μs],str,bool,i64,i64,i64,i64,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,f64
"""1556""","""2972""",2010-01-01 00:01:04.358116,"""7475327""",false,246,610,4563,558,true,false,false,false,false,false,false,false,false,false,false,true,true,false,true,false,false,true,false,-16.61
"""1129""","""102""",2010-01-01 00:02:00.986346,"""7475329""",false,477,1461,3772,303,true,false,false,false,false,false,false,false,false,false,false,true,false,false,true,false,false,true,true,-16.61
"""561""","""4575""",2010-01-01 00:02:10.262363,"""7475328""",false,463,1583,5447,341,true,false,false,false,false,false,false,false,false,false,false,true,false,true,false,false,false,true,true,-16.61
"""848""","""3915""",2010-01-01 00:06:12.236954,"""7475332""",false,500,184,3651,328,true,false,false,false,false,false,false,false,true,false,false,false,false,false,true,false,false,true,true,-16.61
"""1807""","""165""",2010-01-01 00:07:06.349402,"""7475333""",false,445,731,1519,335,true,false,false,false,false,false,false,false,false,false,false,true,true,false,true,false,false,true,false,-16.61


In [25]:
Metrics_OHE_fraud_data_df.select(
    pl.len().alias("rows"),
    pl.col("id").n_unique().alias("unique_ids")
).collect()

rows,unique_ids
u32,u32
3121274,3121274


In [26]:
Metrics_OHE_fraud_data_df.select(
    pl.all().null_count()
).collect()

client_id,card_id,date,id,target,agemonth_at_event,days_acct_open,days_expires,retirement_years,No_Errors,Bad_Card_Number,Bad_CVV,Bad_Expiration,Bad_PIN,Bad_Zipcode,Insufficient_Balance,Technical_Glitch,Visa,Discover,Amex,Mastercard,Debit_Prepaid,Credit,Debit,Online_Transaction,Chip_Transaction,Swipe_Transaction,Male,Log2_risk_factor
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [27]:
(
    Metrics_OHE_fraud_data_df
    .sort(["client_id", "card_id", "date", "id"])
    .sink_parquet(f"{path}silver/Polars/Metrics_OHE_fraud_data.parquet")
)

In [3]:
import polars as pl
import polars.selectors as cs
path = "/home/jovyan/work/data/"

In [4]:
Metrics_OHE_fraud_data_df = pl.scan_parquet(f"{path}silver/Polars/Metrics_OHE_fraud_data.parquet")
amount_features_fraud_data_df = pl.scan_parquet(f"{path}silver/Polars/amount_features_fraud_data.parquet")
spatial_features_fraud_data_df = pl.scan_parquet(f"{path}silver/Polars/spatial_features_fraud_data.parquet")
temporal_features_fraud_data_df = pl.scan_parquet(f"{path}silver/Polars/temporal_features_fraud_data.parquet")
time_diff_features_fraud_data_df = pl.scan_parquet(f"{path}silver/Polars/time_diff_features_fraud_data.parquet")

feature_complete_fraud_data_df = (
    Metrics_OHE_fraud_data_df
    .join(amount_features_fraud_data_df, on=["client_id", "card_id", "date", "id"], how="left")
    .join(spatial_features_fraud_data_df, on=["client_id", "card_id", "date", "id"], how="left")
    .join(temporal_features_fraud_data_df, on=["client_id", "card_id", "date", "id"], how="left")
    .join(time_diff_features_fraud_data_df, on=["client_id", "card_id", "date", "id"], how="left")
)

feature_complete_fraud_data_df.show(5)

client_id,card_id,date,id,target,agemonth_at_event,days_acct_open,days_expires,retirement_years,No_Errors,Bad_Card_Number,Bad_CVV,Bad_Expiration,Bad_PIN,Bad_Zipcode,Insufficient_Balance,Technical_Glitch,Visa,Discover,Amex,Mastercard,Debit_Prepaid,Credit,Debit,Online_Transaction,Chip_Transaction,Swipe_Transaction,Male,Log2_risk_factor,amount,credit_limit,yearly_income,total_debt,sum_amount_24h,mean_log_amount_7d,stddev_log_amount_7d,sum_amount_7d,…,ic_lower_speed_95,ic_upper_speed_95,ic_lower_speed_99,ic_upper_speed_99,is_tdiffm_in_IC90,is_tdiffm_in_IC95,is_tdiffm_in_IC99,hour_of_day,theta,sin_theta,cos_theta,N_7d,day_of_week,weekend,mu_vM_7d,sigma_vM_7d,ic_lower_vm_90_7d,ic_upper_vm_90_7d,ic_lower_vm_95_7d,ic_upper_vm_95_7d,ic_lower_vm_99_7d,ic_upper_vm_99_7d,is_time_in_IC_90,is_time_in_IC_95,is_time_in_IC_99,log_time_diff_m,mean_time_diff_m,stddev_time_diff_m,ic_lower_tdiffm_90,ic_upper_tdiffm_90,ic_lower_tdiffm_95,ic_upper_tdiffm_95,ic_lower_tdiffm_99,ic_upper_tdiffm_99,is_tdiffm_in_IC90_right,is_tdiffm_in_IC95_right,is_tdiffm_in_IC99_right
str,str,datetime[μs],str,bool,i64,i64,i64,i64,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,bool,bool,bool,f64,f64,f64,f64,u32,i8,bool,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool
"""0""","""1271""",2013-07-20 13:18:00.001909,"""13051758""",false,328,900,2811,500,true,false,false,false,false,false,false,false,false,false,false,true,false,false,true,false,false,true,true,2.45,44.17,31490.0,59613.0,36199.0,90.3,3.641438,0.859551,551.63,…,0.0,0.0,0.0,0.0,true,true,true,13.3,-2.801253,-0.333807,-0.942641,14,6,true,-1.651931,1.195052,-2.217551,-1.08631,-2.341933,-0.961928,-2.614025,-0.689837,false,false,false,6.911747,5.48505,2.067322,4.506582,6.463517,4.291413,6.678686,3.820722,7.149377,false,false,true
"""0""","""1271""",2013-07-19 13:23:00.001906,"""13047138""",false,328,899,2812,500,true,false,false,false,false,false,false,false,false,false,false,true,false,false,true,false,false,true,true,0.5,42.52,31490.0,59613.0,36199.0,52.38,3.660233,0.861347,562.84,…,0.0,0.0,0.0,0.0,true,true,true,13.383333,-2.779437,-0.354291,-0.935135,14,5,true,-1.663538,1.198727,-2.230898,-1.096179,-2.355663,-0.971414,-2.628591,-0.698486,false,false,false,7.248504,5.526262,1.964124,4.596639,6.455886,4.392211,6.660314,3.945016,7.107509,false,false,false
"""0""","""1271""",2013-07-23 06:42:00.001911,"""13063377""",false,328,903,2808,500,true,false,false,false,false,false,false,false,false,false,false,true,false,false,true,false,false,true,true,-3.73,14.5,31490.0,59613.0,36199.0,47.47,3.814873,0.60868,434.15,…,0.0,0.0,0.0,0.0,true,true,true,6.7,1.754056,0.983255,-0.182236,11,2,false,-2.445773,1.100984,-3.047436,-1.84411,3.097761,-1.706122,2.785344,-1.393704,false,false,false,6.957497,5.535058,2.399648,4.223704,6.846412,3.922953,7.147164,3.242022,7.828094,false,true,true
"""0""","""1271""",2013-07-23 14:15:00.001912,"""13065451""",false,328,903,2808,500,true,false,false,false,false,false,false,false,false,false,false,true,false,false,true,true,false,false,true,-0.81,16.73,31490.0,59613.0,36199.0,14.5,3.70915,0.687576,400.06,…,0.0,0.0,0.0,0.0,true,true,true,14.25,-2.552544,-0.55557,-0.83147,11,2,false,-2.56345,1.320764,2.997968,-1.841682,2.832434,-1.676148,2.457651,-1.301365,true,true,true,6.118097,5.548354,2.407817,4.232536,6.864172,3.93076,7.165948,3.247512,7.849196,true,true,true
"""0""","""1271""",2013-07-19 20:35:00.001908,"""13048785""",false,328,899,2812,500,true,false,false,false,false,false,false,false,false,false,false,true,false,false,true,false,false,true,true,-0.18,37.51,31490.0,59613.0,36199.0,52.79,3.667118,0.831563,605.25,…,0.004867,0.004867,0.004867,0.004867,true,true,true,20.583333,-0.894481,-0.779884,0.625923,15,5,true,-1.777568,1.205201,-2.325655,-1.22948,-2.444986,-1.110149,-2.703907,-0.851229,false,false,true,2.833213,5.643853,1.944779,4.759429,

In [5]:
feature_df = feature_complete_fraud_data_df.collect()

In [6]:
feature_df.select(
    pl.len().alias("rows"),
    pl.col("id").n_unique().alias("unique_ids")
)

rows,unique_ids
u32,u32
3121274,3121274


In [7]:
analysis = pl.concat(
    [feature_df.select(
        pl.lit(col).alias("COLUMN"),
        pl.col(col).is_infinite().sum().alias("INF"),
        pl.col(col).is_null().sum().alias("NULL"),
     )
     for col in feature_df.select(cs.numeric()).columns
    ]
)

with pl.Config(tbl_rows=-1, fmt_str_lengths=100):
    print(analysis)

shape: (59, 3)
┌──────────────────────────────┬────────┬────────┐
│ COLUMN                       ┆ INF    ┆ NULL   │
│ ---                          ┆ ---    ┆ ---    │
│ str                          ┆ u32    ┆ u32    │
╞══════════════════════════════╪════════╪════════╡
│ agemonth_at_event            ┆ 0      ┆ 0      │
│ days_acct_open               ┆ 0      ┆ 0      │
│ days_expires                 ┆ 0      ┆ 0      │
│ retirement_years             ┆ 0      ┆ 0      │
│ Log2_risk_factor             ┆ 0      ┆ 0      │
│ amount                       ┆ 0      ┆ 0      │
│ credit_limit                 ┆ 0      ┆ 0      │
│ yearly_income                ┆ 0      ┆ 0      │
│ total_debt                   ┆ 0      ┆ 0      │
│ sum_amount_24h               ┆ 0      ┆ 0      │
│ mean_log_amount_7d           ┆ 0      ┆ 0      │
│ stddev_log_amount_7d         ┆ 0      ┆ 0      │
│ sum_amount_7d                ┆ 0      ┆ 0      │
│ acceleration_ratio_24h_vs_7d ┆ 102    ┆ 0      │
│ ratio_24_credi

In [8]:
INF_cols = [
    "acceleration_ratio_24h_vs_7d",
    "ratio_24_credit_limit",
    "ratio_24_total_debt",
    "speed_km_s",
    "mean_speed_km_s"
]

max_values = (
    feature_df
    .select([pl.col(col).filter(~pl.col(col).is_infinite()).max().alias(col)
             for col in INF_cols
    ])
    .row(0, named=True)
)

feature_df = feature_df.with_columns(
    [pl.when(pl.col(col).is_infinite()).then(True).otherwise(False).alias(f"{col}_is_inf")
    for col in INF_cols
    ] +
    [pl.when(pl.col(col).is_infinite()).then(max_values[col]).otherwise(pl.col(col)).alias(col)
    for col in INF_cols
    ]
)

In [9]:
feature_df.show()

client_id,card_id,date,id,target,agemonth_at_event,days_acct_open,days_expires,retirement_years,No_Errors,Bad_Card_Number,Bad_CVV,Bad_Expiration,Bad_PIN,Bad_Zipcode,Insufficient_Balance,Technical_Glitch,Visa,Discover,Amex,Mastercard,Debit_Prepaid,Credit,Debit,Online_Transaction,Chip_Transaction,Swipe_Transaction,Male,Log2_risk_factor,amount,credit_limit,yearly_income,total_debt,sum_amount_24h,mean_log_amount_7d,stddev_log_amount_7d,sum_amount_7d,…,is_tdiffm_in_IC95,is_tdiffm_in_IC99,hour_of_day,theta,sin_theta,cos_theta,N_7d,day_of_week,weekend,mu_vM_7d,sigma_vM_7d,ic_lower_vm_90_7d,ic_upper_vm_90_7d,ic_lower_vm_95_7d,ic_upper_vm_95_7d,ic_lower_vm_99_7d,ic_upper_vm_99_7d,is_time_in_IC_90,is_time_in_IC_95,is_time_in_IC_99,log_time_diff_m,mean_time_diff_m,stddev_time_diff_m,ic_lower_tdiffm_90,ic_upper_tdiffm_90,ic_lower_tdiffm_95,ic_upper_tdiffm_95,ic_lower_tdiffm_99,ic_upper_tdiffm_99,is_tdiffm_in_IC90_right,is_tdiffm_in_IC95_right,is_tdiffm_in_IC99_right,acceleration_ratio_24h_vs_7d_is_inf,ratio_24_credit_limit_is_inf,ratio_24_total_debt_is_inf,speed_km_s_is_inf,mean_speed_km_s_is_inf
str,str,datetime[μs],str,bool,i64,i64,i64,i64,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,bool,bool,f64,f64,f64,f64,u32,i8,bool,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool
"""0""","""1271""",2013-07-19 13:23:00.001906,"""13047138""",false,328,899,2812,500,true,false,false,false,false,false,false,false,false,false,false,true,false,false,true,false,false,true,true,0.5,42.52,31490.0,59613.0,36199.0,52.38,3.660233,0.861347,562.84,…,true,true,13.383333,-2.779437,-0.354291,-0.935135,14,5,true,-1.663538,1.198727,-2.230898,-1.096179,-2.355663,-0.971414,-2.628591,-0.698486,false,false,false,7.248504,5.526262,1.964124,4.596639,6.455886,4.392211,6.660314,3.945016,7.107509,false,false,false,false,false,false,false,false
"""0""","""1271""",2013-07-19 20:35:00.001908,"""13048785""",false,328,899,2812,500,true,false,false,false,false,false,false,false,false,false,false,true,false,false,true,false,false,true,true,-0.18,37.51,31490.0,59613.0,36199.0,52.79,3.667118,0.831563,605.25,…,true,true,20.583333,-0.894481,-0.779884,0.625923,15,5,true,-1.777568,1.205201,-2.325655,-1.22948,-2.444986,-1.110149,-2.703907,-0.851229,false,false,true,2.833213,5.643853,1.944779,4.759429,6.528277,4.566871,6.720836,4.149062,7.138644,false,false,false,false,false,false,false,false
"""0""","""1271""",2013-07-20 13:18:00.001909,"""13051758""",false,328,900,2811,500,true,false,false,false,false,false,false,false,false,false,false,true,false,false,true,false,false,true,true,2.45,44.17,31490.0,59613.0,36199.0,90.3,3.641438,0.859551,551.63,…,true,true,13.3,-2.801253,-0.333807,-0.942641,14,6,true,-1.651931,1.195052,-2.217551,-1.08631,-2.341933,-0.961928,-2.614025,-0.689837,false,false,false,6.911747,5.48505,2.067322,4.506582,6.463517,4.291413,6.678686,3.820722,7.149377,false,false,true,false,false,false,false,false
"""0""","""1271""",2013-07-23 06:42:00.001911,"""13063377""",false,328,903,2808,500,true,false,false,false,false,false,false,false,false,false,false,true,false,false,true,false,false,true,true,-3.73,14.5,31490.0,59613.0,36199.0,47.47,3.814873,0.60868,434.15,…,true,true,6.7,1.754056,0.983255,-0.182236,11,2,false,-2.445773,1.100984,-3.047436,-1.84411,3.097761,-1.706122,2.785344,-1.393704,false,false,false,6.957497,5.535058,2.399648,4.223704,6.846412,3.922953,7.147164,3.242022,7.828094,false,true,true,false,false,false,false,false
"""0""","""1271""",2013-07-23 14:15:00.001912,"""13065451""",false,328,903,2808,500,true,false,false,false,false,false,false,false,false,false,false,true,false,false,true,true,false,false,true,-0.81,16.73,31490.0,59613.0,36199.0,14.5,3.70915,0.687576,400.06,…,true,true,14.25,-2.552544,-0.55557,-0.83147,11,2,false,-2.56345,1.320764,2.997968,-1.841682,2.832434,-1.676148,2.457651,-1.301365,true,

In [10]:
analysis_2 = pl.concat(
    [feature_df.select(
        pl.lit(col).alias("COLUMN"),
        pl.col(col).is_infinite().sum().alias("INF"),
        pl.col(col).is_null().sum().alias("NULL"),
     )
     for col in feature_df.select(cs.numeric()).columns
    ]
)

with pl.Config(tbl_rows=-1, fmt_str_lengths=100):
    print(analysis_2)

shape: (59, 3)
┌──────────────────────────────┬─────┬────────┐
│ COLUMN                       ┆ INF ┆ NULL   │
│ ---                          ┆ --- ┆ ---    │
│ str                          ┆ u32 ┆ u32    │
╞══════════════════════════════╪═════╪════════╡
│ agemonth_at_event            ┆ 0   ┆ 0      │
│ days_acct_open               ┆ 0   ┆ 0      │
│ days_expires                 ┆ 0   ┆ 0      │
│ retirement_years             ┆ 0   ┆ 0      │
│ Log2_risk_factor             ┆ 0   ┆ 0      │
│ amount                       ┆ 0   ┆ 0      │
│ credit_limit                 ┆ 0   ┆ 0      │
│ yearly_income                ┆ 0   ┆ 0      │
│ total_debt                   ┆ 0   ┆ 0      │
│ sum_amount_24h               ┆ 0   ┆ 0      │
│ mean_log_amount_7d           ┆ 0   ┆ 0      │
│ stddev_log_amount_7d         ┆ 0   ┆ 0      │
│ sum_amount_7d                ┆ 0   ┆ 0      │
│ acceleration_ratio_24h_vs_7d ┆ 0   ┆ 0      │
│ ratio_24_credit_limit        ┆ 0   ┆ 0      │
│ ratio_24_yearly_income 

In [12]:
(
    feature_df
    .sort(["client_id", "card_id", "date", "id"])
    .write_parquet(f"{path}gold/Polars/feature_complete_fraud_data.parquet")
)